In [481]:
import numpy as np
import pandas as pd
from datetime import date, timedelta

In [482]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.precision', 3)

In [483]:
player_stats = pd.read_excel('C:/Users/bdoc1/OneDrive/NBA/2027 NBA Season.xlsm', sheet_name = 'PlayerGameStats')
shooting_stats = pd.read_excel('C:/Users/bdoc1/OneDrive/NBA/2027 NBA Season.xlsm', sheet_name = 'PlayerShootingStats')
lineup_stats = pd.read_excel('C:/Users/bdoc1/OneDrive/NBA/2027 NBA Season.xlsm', sheet_name = 'LineupStats')
opp_lineup_stats = pd.read_excel('C:/Users/bdoc1/OneDrive/NBA/2027 NBA Season.xlsm', sheet_name = 'OppLineupStats')
schedule = pd.read_excel('C:/Users/bdoc1/OneDrive/NBA/2027 NBA Season.xlsm', sheet_name = 'Schedule')
glossary = pd.read_excel('C:/Users/bdoc1/OneDrive/NBA/2027 NBA Season.xlsm', sheet_name = 'TeamGlossary')

In [484]:
player_stats = player_stats.merge(
    glossary[['NBA', 'Display']],
    left_on = 'Team',
    right_on = 'NBA',
    indicator=False
)
player_stats = player_stats.rename(columns={'Team':'Team Initial', 'Display':'Team'})
player_stats.drop(columns='NBA', inplace = True)

In [485]:
shooting_stats['Restricted Area FGM'] = shooting_stats['Restricted Area FGM'].replace('-', 0).astype('int64')
shooting_stats['Restricted Area FGA'] = shooting_stats['Restricted Area FGA'].replace('-', 0).astype('int64')
shooting_stats['In The Paint (Non-RA) FGM'] = shooting_stats['In The Paint (Non-RA) FGM'].replace('-', 0).astype('int64')
shooting_stats['In The Paint (Non-RA) FGA'] = shooting_stats['In The Paint (Non-RA) FGA'].replace('-', 0).astype('int64')
shooting_stats['Mid-Range FGM'] = shooting_stats['Mid-Range FGM'].replace('-', 0).astype('int64')
shooting_stats['Mid-Range FGA'] = shooting_stats['Mid-Range FGA'].replace('-', 0).astype('int64')
shooting_stats['Corner  FGM'] = shooting_stats['Corner  FGM'].replace('-', 0).astype('int64')
shooting_stats['Corner  FGA'] = shooting_stats['Corner  FGA'].replace('-', 0).astype('int64')
shooting_stats['Above the Break 3 FGM'] = shooting_stats['Above the Break 3 FGM'].replace('-', 0).astype('int64')
shooting_stats['Above the Break 3 FGA'] = shooting_stats['Above the Break 3 FGA'].replace('-', 0).astype('int64')

C:\Users\bdoc1\AppData\Local\Temp\ipykernel_28352\2350697142.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  shooting_stats['Restricted Area FGM'] = shooting_stats['Restricted Area FGM'].replace('-', 0).astype('int64')
C:\Users\bdoc1\AppData\Local\Temp\ipykernel_28352\2350697142.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  shooting_stats['Restricted Area FGA'] = shooting_stats['Restricted Area FGA'].replace('-', 0).astype('int64')
C:\Users\bdoc1\AppData\Local\Temp\ipykernel_28352\2350697142.py:3: FutureWarning: Downcast

In [486]:
game_and_shooting = player_stats.merge(
    shooting_stats[['Player', 'Date', 'Restricted Area FGM', 'Restricted Area FGA', 'Restricted Area FG%', 'In The Paint (Non-RA) FGM', 
                    'In The Paint (Non-RA) FGA', 'In The Paint (Non-RA) FG%', 'Mid-Range FGM', 'Mid-Range FGA', 'Mid-Range FG%', 
                    'Left Corner 3 FGM', 'Left Corner 3 FGA', 'Left Corner 3 FG%', 'Right Corner 3 FGM', 'Right Corner 3 FGA', 
                    'Right Corner 3 FG%', 'Corner  FGM', 'Corner  FGA', 'Corner  FG%', 'Above the Break 3 FGM', 'Above the Break 3 FGA',
                    'Above the Break 3 FG%']], on=['Player', 'Date'], indicator=True
)

In [487]:
game_and_shooting = game_and_shooting.drop(columns='_merge')
game_and_shooting['initial_last'] = game_and_shooting['Player'].str.split(' ', n=1, expand=True)[0].str[0]+'.'+' '+game_and_shooting['Player'].str.split(' ', n=1, expand=True)[1]

In [488]:
full_lineup_stats = lineup_stats.merge(
    opp_lineup_stats[['Date', 'Lineups', 'Opp FGM', 'Opp FGA', 'Opp FG%', 'Opp 3PM', 'Opp 3PA', 'Opp 3P%', 'Opp FTM', 'Opp FTA', 'Opp FT%',
                   'Opp OREB', 'Opp DREB', 'Opp REB', 'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 'Opp BLKA', 'Opp PF', 'Opp PFD', 'Opp PTS']], 
                    on=['Date', 'Lineups'], indicator=True
)
full_lineup_stats = full_lineup_stats.merge(
    glossary[['NBA', 'Display']],
    left_on = 'Team',
    right_on = 'NBA',
    indicator=False
)
full_lineup_stats = full_lineup_stats.rename(columns={'Team':'Team Initial', 'Display':'Team'})
full_lineup_stats.drop(columns='NBA', inplace = True)

In [489]:
full_lineup_stats[['Player 1', 'Player 2', 'Player 3', 'Player 4', 'Player 5']] = full_lineup_stats['Lineups'].str.split(' - ', n=4, expand=True)

In [490]:
full_lineup_stats['Without Player 1'] = full_lineup_stats['Player 2'] +' - '+ full_lineup_stats['Player 3'] +' - '+ full_lineup_stats['Player 4'] +' - '+ full_lineup_stats['Player 5']
full_lineup_stats['Without Player 2'] = full_lineup_stats['Player 1'] +' - '+ full_lineup_stats['Player 3'] +' - '+ full_lineup_stats['Player 4'] +' - '+ full_lineup_stats['Player 5']
full_lineup_stats['Without Player 3'] = full_lineup_stats['Player 1'] +' - '+ full_lineup_stats['Player 2'] +' - '+ full_lineup_stats['Player 4'] +' - '+ full_lineup_stats['Player 5']
full_lineup_stats['Without Player 4'] = full_lineup_stats['Player 1'] +' - '+ full_lineup_stats['Player 2'] +' - '+ full_lineup_stats['Player 3'] +' - '+ full_lineup_stats['Player 5']
full_lineup_stats['Without Player 5'] = full_lineup_stats['Player 1'] +' - '+ full_lineup_stats['Player 2'] +' - '+ full_lineup_stats['Player 3'] +' - '+ full_lineup_stats['Player 4']

In [491]:
player_by_player_lineup_1 = full_lineup_stats.melt(
    id_vars = ['Date', 'Team', 'Player 1', 'GP', 'Min', 'PTS', 'FGM', 'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 'OREB', 'DREB', 'REB', 
               'AST', 'TOV', 'STL', 'BLK', 'PF', 'PFD', '+/-', 'Opp FGM', 'Opp FGA', 'Opp FG%', 'Opp 3PM', 'Opp 3PA', 'Opp 3P%', 'Opp FTM', 
               'Opp FTA', 'Opp FT%', 'Opp OREB', 'Opp DREB', 'Opp REB', 'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 'Opp PTS', 'Lineups', 
               'Without Player 1'])
player_lineup_1 = player_by_player_lineup_1[player_by_player_lineup_1['variable'] == '_merge']
player_lineup_1 = player_lineup_1.rename(columns={'Player 1':'Player', 'Without Player 1':'Without Player'})
player_by_player_lineup_2 = full_lineup_stats.melt(
    id_vars = ['Date', 'Team', 'Player 2', 'GP', 'Min', 'PTS', 'FGM', 'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 'OREB', 'DREB', 'REB', 
               'AST', 'TOV', 'STL', 'BLK', 'PF', 'PFD', '+/-', 'Opp FGM', 'Opp FGA', 'Opp FG%', 'Opp 3PM', 'Opp 3PA', 'Opp 3P%', 'Opp FTM', 
               'Opp FTA', 'Opp FT%', 'Opp OREB', 'Opp DREB', 'Opp REB', 'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 'Opp PTS', 'Lineups',
              'Without Player 2'])
player_lineup_2 = player_by_player_lineup_2[player_by_player_lineup_2['variable'] == '_merge']
player_lineup_2 = player_lineup_2.rename(columns={'Player 2':'Player', 'Without Player 2':'Without Player'})
player_by_player_lineup_3 = full_lineup_stats.melt(
    id_vars = ['Date', 'Team', 'Player 3', 'GP', 'Min', 'PTS', 'FGM', 'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 'OREB', 'DREB', 'REB', 
               'AST', 'TOV', 'STL', 'BLK', 'PF', 'PFD', '+/-', 'Opp FGM', 'Opp FGA', 'Opp FG%', 'Opp 3PM', 'Opp 3PA', 'Opp 3P%', 'Opp FTM', 
               'Opp FTA', 'Opp FT%', 'Opp OREB', 'Opp DREB', 'Opp REB', 'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 'Opp PTS', 'Lineups',
              'Without Player 3'])
player_lineup_3 = player_by_player_lineup_3[player_by_player_lineup_3['variable'] == '_merge']
player_lineup_3 = player_lineup_3.rename(columns={'Player 3':'Player', 'Without Player 3':'Without Player'})
player_by_player_lineup_4 = full_lineup_stats.melt(
    id_vars = ['Date', 'Team', 'Player 4', 'GP', 'Min', 'PTS', 'FGM', 'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 'OREB', 'DREB', 'REB', 
               'AST', 'TOV', 'STL', 'BLK', 'PF', 'PFD', '+/-', 'Opp FGM', 'Opp FGA', 'Opp FG%', 'Opp 3PM', 'Opp 3PA', 'Opp 3P%', 'Opp FTM', 
               'Opp FTA', 'Opp FT%', 'Opp OREB', 'Opp DREB', 'Opp REB', 'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 'Opp PTS', 'Lineups',
              'Without Player 4'])
player_lineup_4 = player_by_player_lineup_4[player_by_player_lineup_4['variable'] == '_merge']
player_lineup_4 = player_lineup_4.rename(columns={'Player 4':'Player', 'Without Player 4':'Without Player'})
player_by_player_lineup_5 = full_lineup_stats.melt(
    id_vars = ['Date', 'Team', 'Player 5', 'GP', 'Min', 'PTS', 'FGM', 'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', 'FTA', 'FT%', 'OREB', 'DREB', 'REB', 
               'AST', 'TOV', 'STL', 'BLK', 'PF', 'PFD', '+/-', 'Opp FGM', 'Opp FGA', 'Opp FG%', 'Opp 3PM', 'Opp 3PA', 'Opp 3P%', 'Opp FTM', 
               'Opp FTA', 'Opp FT%', 'Opp OREB', 'Opp DREB', 'Opp REB', 'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 'Opp PTS', 'Lineups',
              'Without Player 5'])
player_lineup_5 = player_by_player_lineup_5[player_by_player_lineup_5['variable'] == '_merge']
player_lineup_5 = player_lineup_5.rename(columns={'Player 5':'Player', 'Without Player 5':'Without Player'})

In [492]:
player_oncourt = pd.concat([player_lineup_1, player_lineup_2, player_lineup_3, player_lineup_4, player_lineup_5], axis=0)

In [493]:
player_oncourt = player_oncourt.rename(columns={'PTS':'OnCourt PTS', 'FGM':'OnCourt FGM', 'FGA':'OnCourt FGA', 'FG%':'OnCourt FG%',
                                               '3PM':'OnCourt 3PM', '3PA':'OnCourt 3PA', '3P%':'OnCourt 3P%', 'FTM':'OnCourt FTM',
                                               'FTA':'OnCourt FTA', 'FT%':'OnCourt FT%', 'OREB':'OnCourt OREB', 'DREB':'OnCourt DREB', 
                                                'REB':'OnCourt REB', 'AST':'OnCourt AST', 'TOV':'OnCourt TOV', 'STL':'OnCourt STL',
                                                'BLK':'OnCourt BLK', 'PF':'OnCourt PF', 'PFD':'OnCourt PFD', 'Opp FGM':'OnCourt Opp FGM', 
                                                'Opp FGA':'OnCourt Opp FGA', 'Opp FG%':'OnCourt Opp FG%', 'Opp 3PM':'OnCourt Opp 3PM', 
                                               'Opp 3PA':'OnCourt Opp 3PA', 'Opp 3P%':'OnCourt Opp 3P%', 'Opp FTM':'OnCourt Opp FTM', 
                                                'Opp FTA':'OnCourt Opp FTA', 'Opp FT%':'OnCourt Opp FT%', 'Opp OREB':'OnCourt Opp OREB',
                                               'Opp DREB':'OnCourt Opp DREB', 'Opp REB':'OnCourt Opp REB', 'Opp AST':'OnCourt Opp AST', 
                                                'Opp TOV':'OnCourt Opp TOV', 'Opp STL':'OnCourt Opp STL', 'Opp BLK':'OnCourt Opp BLK', 
                                                'Opp PTS':'OnCourt Opp PTS'})

In [494]:
lineup_on_court = player_oncourt.groupby(['Date', 'Team', 'Without Player'])[['Min', 'OnCourt PTS', 'OnCourt FGM',
        'OnCourt FGA', 'OnCourt 3PM', 'OnCourt 3PA', 'OnCourt FTM', 'OnCourt FTA', 'OnCourt OREB', 'OnCourt DREB', 'OnCourt REB', 'OnCourt AST',
        'OnCourt TOV', 'OnCourt STL', 'OnCourt BLK', 'OnCourt PF', 'OnCourt PFD', '+/-', 'OnCourt Opp FGM', 'OnCourt Opp FGA', 'OnCourt Opp 3PM',
        'OnCourt Opp 3PA', 'OnCourt Opp FTM', 'OnCourt Opp FTA', 'OnCourt Opp OREB', 'OnCourt Opp DREB', 'OnCourt Opp REB', 'OnCourt Opp AST',
        'OnCourt Opp TOV', 'OnCourt Opp STL', 'OnCourt Opp BLK', 'OnCourt Opp PTS']].sum().reset_index()

In [495]:
lineup_on_court = lineup_on_court.rename(columns={'Min':'4-Man Min', 'OnCourt PTS':'4-Man PTS', 'OnCourt FGM':'4-Man FGM',
                                'OnCourt FGA':'4-Man FGA', 'OnCourt 3PM':'4-Man 3PM', 'OnCourt 3PA':'4-Man 3PA', 'OnCourt FTM':'4-Man FTM',
                                'OnCourt FTA':'4-Man FTA', 'OnCourt OREB':'4-Man OREB', 'OnCourt DREB':'4-Man DREB', 'OnCourt REB':'4-Man REB',
                                'OnCourt AST':'4-Man AST', 'OnCourt TOV':'4-Man TOV', 'OnCourt STL':'4-Man STL', 'OnCourt BLK':'4-Man BLK',
                                'OnCourt PF':'4-Man PF', 'OnCourt PFD':'4-Man PFD', '+/-':'4-Man +/-', 'OnCourt Opp FGM':'4-Man Opp FGM',
                                'OnCourt Opp FGA':'4-Man Opp FGA', 'OnCourt Opp 3PM':'4-Man Opp 3PM', 'OnCourt Opp 3PA':'4-Man Opp 3PA',
                                'OnCourt Opp FTM':'4-Man Opp FTM', 'OnCourt Opp FTA':'4-Man Opp FTA', 'OnCourt Opp OREB':'4-Man Opp OREB',
                                'OnCourt Opp DREB':'4-Man Opp DREB', 'OnCourt Opp REB':'4-Man Opp REB', 'OnCourt Opp AST':'4-Man Opp AST',
                                'OnCourt Opp TOV':'4-Man Opp TOV', 'OnCourt Opp STL':'4-Man Opp STL', 'OnCourt Opp BLK':'4-Man Opp BLK',
                                'OnCourt Opp PTS':'4-Man Opp PTS'})

In [496]:
player_oncourt_fourman = player_oncourt.merge(
    lineup_on_court[['Date', 'Team', 'Without Player', '4-Man Min', '4-Man PTS', '4-Man FGM',
       '4-Man FGA', '4-Man 3PM', '4-Man 3PA', '4-Man FTM', '4-Man FTA',
       '4-Man OREB', '4-Man DREB', '4-Man REB', '4-Man AST', '4-Man TOV',
       '4-Man STL', '4-Man BLK', '4-Man PF', '4-Man PFD', '4-Man +/-',
       '4-Man Opp FGM', '4-Man Opp FGA', '4-Man Opp 3PM', '4-Man Opp 3PA',
       '4-Man Opp FTM', '4-Man Opp FTA', '4-Man Opp OREB', '4-Man Opp DREB',
       '4-Man Opp REB', '4-Man Opp AST', '4-Man Opp TOV', '4-Man Opp STL',
       '4-Man Opp BLK', '4-Man Opp PTS']],
    on = ['Date', 'Team', 'Without Player'],
    indicator = True
)

In [497]:
stats = ['PTS', 'FGM', 'FGA', '3PM', '3PA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'PF', 'PFD',
         'Opp PTS', 'Opp FGM', 'Opp FGA', 'Opp 3PM', 'Opp 3PA', 'Opp FTM', 'Opp FTA', 'Opp OREB', 'Opp DREB', 'Opp REB', 'Opp AST', 'Opp TOV',
         'Opp STL', 'Opp BLK', 'Opp PTS']
player_oncourt_fourman['4-Man Without Player Min'] = player_oncourt_fourman['4-Man Min'] - player_oncourt_fourman['Min']
player_oncourt_fourman['4-Man Without Player +/-'] = player_oncourt_fourman['4-Man +/-'] - player_oncourt_fourman['+/-']
for i in stats:
    player_oncourt_fourman[f'4-Man Without Player {i}'] = player_oncourt_fourman[f'4-Man {i}'] - player_oncourt_fourman[f'OnCourt {i}']

In [498]:
full_player_on_court = player_oncourt_fourman.groupby(['Date', 'Team', 'Player'])[['Min', 'OnCourt PTS', 'OnCourt FGM',
       'OnCourt FGA', 'OnCourt 3PM', 'OnCourt 3PA', 'OnCourt FTM', 'OnCourt FTA', 'OnCourt OREB', 'OnCourt DREB', 'OnCourt REB', 'OnCourt AST', 
    'OnCourt TOV', 'OnCourt STL', 'OnCourt BLK', 'OnCourt PF', 'OnCourt PFD', '+/-', 'OnCourt Opp FGM', 'OnCourt Opp FGA', 'OnCourt Opp 3PM', 
    'OnCourt Opp 3PA', 'OnCourt Opp FTM', 'OnCourt Opp FTA', 'OnCourt Opp OREB', 'OnCourt Opp DREB', 'OnCourt Opp REB', 'OnCourt Opp AST', 
    'OnCourt Opp TOV', 'OnCourt Opp STL', 'OnCourt Opp BLK', 'OnCourt Opp PTS', '4-Man Min', '4-Man PTS', '4-Man FGM', '4-Man FGA', '4-Man 3PM',
    '4-Man 3PA', '4-Man FTM', '4-Man FTA', '4-Man OREB', '4-Man DREB', '4-Man REB', '4-Man AST', '4-Man TOV', '4-Man STL', '4-Man BLK',
    '4-Man PF', '4-Man PFD', '4-Man +/-', '4-Man Opp FGM', '4-Man Opp FGA', '4-Man Opp 3PM', '4-Man Opp 3PA', '4-Man Opp FTM', '4-Man Opp FTA',
    '4-Man Opp OREB', '4-Man Opp DREB', '4-Man Opp REB', '4-Man Opp AST', '4-Man Opp TOV', '4-Man Opp STL', '4-Man Opp BLK', '4-Man Opp PTS',
    '4-Man Without Player Min', '4-Man Without Player PTS', '4-Man Without Player FGM', '4-Man Without Player FGA',
       '4-Man Without Player 3PM', '4-Man Without Player 3PA', '4-Man Without Player FTM', '4-Man Without Player FTA',
       '4-Man Without Player OREB', '4-Man Without Player DREB', '4-Man Without Player REB', '4-Man Without Player AST',
       '4-Man Without Player TOV', '4-Man Without Player STL', '4-Man Without Player BLK', '4-Man Without Player PF',
       '4-Man Without Player PFD', '4-Man Without Player +/-', '4-Man Without Player Opp PTS','4-Man Without Player Opp FGM', 
        '4-Man Without Player Opp FGA', '4-Man Without Player Opp 3PM', '4-Man Without Player Opp 3PA','4-Man Without Player Opp FTM', 
        '4-Man Without Player Opp FTA', '4-Man Without Player Opp OREB', '4-Man Without Player Opp DREB','4-Man Without Player Opp REB', 
        '4-Man Without Player Opp AST', '4-Man Without Player Opp TOV', '4-Man Without Player Opp STL',
        '4-Man Without Player Opp BLK']].sum().reset_index()

In [499]:
game_shooting_lineup = game_and_shooting.merge(
    full_player_on_court[['Date', 'Team', 'Player', 'OnCourt PTS', 'OnCourt FGM', 'OnCourt FGA', 'OnCourt 3PM', 'OnCourt 3PA', 'OnCourt FTM',
                          'OnCourt FTA', 'OnCourt OREB', 'OnCourt DREB', 'OnCourt REB', 'OnCourt AST', 'OnCourt TOV', 'OnCourt STL', 'OnCourt BLK',
                          'OnCourt PF', 'OnCourt PFD', 'OnCourt Opp FGM', 'OnCourt Opp FGA', 'OnCourt Opp 3PM', 'OnCourt Opp 3PA', 'OnCourt Opp FTM', 
                          'OnCourt Opp FTA', 'OnCourt Opp OREB', 'OnCourt Opp DREB', 'OnCourt Opp REB', 'OnCourt Opp AST', 'OnCourt Opp TOV', 
                          'OnCourt Opp STL', 'OnCourt Opp BLK', 'OnCourt Opp PTS', '4-Man Min', '4-Man PTS', '4-Man FGM', '4-Man FGA', '4-Man 3PM',
    '4-Man 3PA', '4-Man FTM', '4-Man FTA', '4-Man OREB', '4-Man DREB', '4-Man REB', '4-Man AST', '4-Man TOV', '4-Man STL', '4-Man BLK',
    '4-Man PF', '4-Man PFD', '4-Man +/-', '4-Man Opp FGM', '4-Man Opp FGA', '4-Man Opp 3PM', '4-Man Opp 3PA', '4-Man Opp FTM', '4-Man Opp FTA',
    '4-Man Opp OREB', '4-Man Opp DREB', '4-Man Opp REB', '4-Man Opp AST', '4-Man Opp TOV', '4-Man Opp STL', '4-Man Opp BLK', '4-Man Opp PTS',
    '4-Man Without Player Min', '4-Man Without Player PTS', '4-Man Without Player FGM', '4-Man Without Player FGA',
       '4-Man Without Player 3PM', '4-Man Without Player 3PA', '4-Man Without Player FTM', '4-Man Without Player FTA',
       '4-Man Without Player OREB', '4-Man Without Player DREB', '4-Man Without Player REB', '4-Man Without Player AST',
       '4-Man Without Player TOV', '4-Man Without Player STL', '4-Man Without Player BLK', '4-Man Without Player PF',
       '4-Man Without Player PFD', '4-Man Without Player +/-', '4-Man Without Player Opp PTS','4-Man Without Player Opp FGM', 
        '4-Man Without Player Opp FGA', '4-Man Without Player Opp 3PM', '4-Man Without Player Opp 3PA','4-Man Without Player Opp FTM', 
        '4-Man Without Player Opp FTA', '4-Man Without Player Opp OREB', '4-Man Without Player Opp DREB','4-Man Without Player Opp REB', 
        '4-Man Without Player Opp AST','4-Man Without Player Opp TOV', '4-Man Without Player Opp STL','4-Man Without Player Opp BLK']], 
    left_on = ['Date', 'Team', 'initial_last'], 
    right_on = ['Date', 'Team', 'Player'], indicator=True
)
game_shooting_lineup.drop(columns='_merge', inplace=True)

In [500]:
team_stats = game_shooting_lineup.groupby(['Date', 'Team'])[['Min', 'PTS', 'FGM', 'FGA', '3PM', '3PA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB', 
                                                             'AST', 'TOV', 'STL', 'BLK', 'PF', 'Restricted Area FGM', 'Restricted Area FGA',
                                                             'In The Paint (Non-RA) FGM', 'In The Paint (Non-RA) FGA', 'Mid-Range FGM', 
                                                             'Mid-Range FGA', 'Corner  FGM', 'Corner  FGA', 'Above the Break 3 FGM',
                                                             'Above the Break 3 FGA']].sum().reset_index()
team_stats['OT'] = np.where(team_stats['Min'] > 242, 'OT', '')

In [501]:
schedule = schedule.merge(
    glossary[['Basketball Reference', 'Display']],
    left_on = 'Visitor/Neutral',
    right_on = 'Basketball Reference',
    indicator = True
)
schedule = schedule.drop(columns=['Basketball Reference', '_merge'])
schedule = schedule.rename(columns={'Display':'Away Initial'})
schedule = schedule.merge(
    glossary[['Basketball Reference', 'Display']],
    left_on = 'Home/Neutral',
    right_on = 'Basketball Reference',
    indicator = True
)
schedule = schedule.drop(columns=['Basketball Reference', '_merge'])
schedule = schedule.rename(columns={'Display':'Home Initial'})

In [502]:
schedule['new_date'] = pd.to_datetime(schedule['Date'], format='%a, %b %d, %Y', errors='coerce')
schedule['Away Initial Date'] = schedule['Away Initial'] +' '+schedule['new_date'].astype('str')
schedule['Home Initial Date'] = schedule['Home Initial'] +' '+schedule['new_date'].astype('str')
team_stats['Team Date'] = team_stats['Team'] +' '+team_stats['Date'].astype('str')
team_stats['H_A'] = np.where(team_stats['Team Date'].isin(schedule['Home Initial Date']), 1, 0)

In [503]:
team_stats_home = team_stats.merge(
    schedule[['Home Initial Date', 'Away Initial']],
    left_on = 'Team Date',
    right_on = 'Home Initial Date',
    indicator = True
)
team_stats_home = team_stats_home.drop(columns=['Home Initial Date', '_merge'])
team_stats_home = team_stats_home.rename(columns={'Away Initial':'Opponent'})
team_stats_away = team_stats.merge(
    schedule[['Away Initial Date', 'Home Initial']],
    left_on = 'Team Date',
    right_on = 'Away Initial Date',
    indicator = True
)
team_stats_away = team_stats_away.drop(columns=['Away Initial Date', '_merge'])
team_stats_away = team_stats_away.rename(columns={'Home Initial':'Opponent'})
team_stats_home_away = pd.concat([team_stats_away, team_stats_home], axis = 0)

In [504]:
opponent_stats = team_stats_home_away.drop(columns=['Team', 'Team Date'])
opponent_stats = opponent_stats.rename(columns={'Opponent':'Team'})

In [505]:
opponent_stats = opponent_stats.rename(columns={'PTS':'Opp PTS', 'FGM':'Opp FGM', 'FGA':'Opp FGA', '3PM':'Opp 3PM', '3PA':'Opp 3PA', 
                                                'FTM':'Opp FTM', 'FTA':'Opp FTA', 'OREB':'Opp OREB', 'DREB':'Opp DREB', 'REB':'Opp REB',
                                                'AST':'Opp AST', 'TOV':'Opp TOV', 'STL':'Opp STL', 'BLK':'Opp BLK', 'PF':'Opp PF',
                                                'Restricted Area FGM':'Opp Restricted Area FGM', 'Restricted Area FGA':'Opp Restricted Area FGA',
                                                'In The Paint (Non-RA) FGM':'Opp In The Paint (Non-RA) FGM', 
                                                'In The Paint (Non-RA) FGA':'Opp In The Paint (Non-RA) FGA', 
                                                'Mid-Range FGM':'Opp Mid-Range FGM', 'Mid-Range FGA':'Opp Mid-Range FGA', 
                                                'Corner  FGM':'Opp Corner FGM', 'Corner  FGA':'Opp Corner FGA', 
                                                'Above the Break 3 FGM':'Opp Above the Break 3 FGM', 
                                                'Above the Break 3 FGA':'Opp Above the Break 3 FGA'})

In [506]:
opponent_stats['Team Date'] = opponent_stats['Team'] +' '+opponent_stats['Date'].astype('str')

In [507]:
team_opp_stats = team_stats_home_away.merge(
    opponent_stats[['Team Date', 'Opp PTS', 'Opp FGM', 'Opp FGA', 'Opp 3PM', 'Opp 3PA', 'Opp FTM', 'Opp FTA', 'Opp OREB', 'Opp DREB', 
                    'Opp REB', 'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 'Opp PF', 'Opp Restricted Area FGM', 'Opp Restricted Area FGA',
                    'Opp In The Paint (Non-RA) FGM', 'Opp In The Paint (Non-RA) FGA', 'Opp Mid-Range FGM', 
                    'Opp Mid-Range FGA', 'Opp Corner FGM', 'Opp Corner FGA', 'Opp Above the Break 3 FGM', 'Opp Above the Break 3 FGA']],
    on = 'Team Date',
    indicator = True
)

In [508]:
team_opp_stats.sort_values(inplace=True, ascending=True, by='Date')

In [509]:
team_opp_stats['+/-'] = team_opp_stats['PTS']-team_opp_stats['Opp PTS']
team_opp_stats['W'] = np.where(team_opp_stats['PTS'] > team_opp_stats['Opp PTS'], 1, 0)
team_opp_stats['L'] = 1-team_opp_stats['W']

In [510]:
team_season_long = team_opp_stats.groupby(['Team'])[['W', 'L', 'PTS', 'FGA', 'FTM', 'FTA', '3PM', '3PA', 'TOV', 'REB', 'Opp REB', 'STL',
                                                     'Opp PTS', 'Opp FGA', 'Opp FTM', 'Opp FTA', 'Opp TOV', 'Restricted Area FGM', 'Opp STL',
                                                     'Restricted Area FGA', 'In The Paint (Non-RA) FGM', 'In The Paint (Non-RA) FGA', 'Mid-Range FGM', 
                                                    'Mid-Range FGA', 'Opp Restricted Area FGM', 'Opp Restricted Area FGA',
                                                    'Opp In The Paint (Non-RA) FGM', 'Opp In The Paint (Non-RA) FGA',
                                                    'Opp Mid-Range FGM', 'Opp Mid-Range FGA','Opp 3PM', 'Opp 3PA']].sum().reset_index()

In [511]:
team_season_long['TS%'] = team_season_long['PTS']/(2*team_season_long['FGA']+.88*team_season_long['FTA'])
team_season_long['TOV%'] = team_season_long['TOV']/(team_season_long['TOV']+team_season_long['FGA']+.44*team_season_long['FTA'])
team_season_long['REB%'] = team_season_long['REB']/(team_season_long['REB']+team_season_long['Opp REB'])
team_season_long['Opp TS%'] = team_season_long['Opp PTS']/(2*team_season_long['Opp FGA']+.88*team_season_long['Opp FTA'])
team_season_long['Opp TOV%'] = team_season_long['Opp TOV']/(team_season_long['Opp TOV']+team_season_long['Opp FGA']+.44*team_season_long['Opp FTA'])
team_season_long['W%'] = team_season_long['W']/(team_season_long['W']+team_season_long['L'])
team_season_long.sort_values(inplace=True, ascending=False, by='W%')

In [512]:
team_opp_stats2 = team_opp_stats.rename(columns={'PTS':'Team PTS', 'FGM':'Team FGM', 'FGA':'Team FGA', '3PM':'Team 3PM', '3PA':'Team 3PA', 
                                                 'FTM':'Team FTM', 'FTA':'Team FTA', 'OREB':'Team OREB', 'DREB':'Team DREB', 'REB':'Team REB', 
                                                 'AST':'Team AST', 'TOV':'Team TOV', 'STL':'Team STL', 'BLK':'Team BLK', 'PF':'Team PF', 
                                                 '+/-':'Team +/-'})

In [513]:
game_shooting_onoff = game_shooting_lineup.merge(
    team_opp_stats2[['Date', 'Team', 'Opponent', 'Team PTS', 'Team FGM', 'Team FGA', 'Team 3PM', 'Team 3PA', 'Team FTM', 'Team FTA', 'Team OREB', 
                     'Team DREB', 'Team REB', 'Team AST', 'Team TOV', 'Team STL', 'Team BLK', 'Team PF', 'Team +/-', 'Opp PTS', 'Opp FGM', 'Opp FGA', 
                     'Opp 3PM','Opp 3PA', 'Opp FTM', 'Opp FTA', 'Opp OREB', 'Opp DREB', 'Opp REB', 'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 
                     'Opp PF', 'H_A']],
    on = ['Date', 'Team'],
    indicator = 'merge_check'
)

In [514]:
game_shooting_onoff['OffCourt PTS'] = game_shooting_onoff['Team PTS'] - game_shooting_onoff['OnCourt PTS']
game_shooting_onoff['OffCourt FGM'] = game_shooting_onoff['Team FGM'] - game_shooting_onoff['OnCourt FGM']
game_shooting_onoff['OffCourt FGA'] = game_shooting_onoff['Team FGA'] - game_shooting_onoff['OnCourt FGA']
game_shooting_onoff['OffCourt 3PM'] = game_shooting_onoff['Team 3PM'] - game_shooting_onoff['OnCourt 3PM']
game_shooting_onoff['OffCourt 3PA'] = game_shooting_onoff['Team 3PA'] - game_shooting_onoff['OnCourt 3PA']
game_shooting_onoff['OffCourt FTM'] = game_shooting_onoff['Team FTM'] - game_shooting_onoff['OnCourt FTM']
game_shooting_onoff['OffCourt FTA'] = game_shooting_onoff['Team FTA'] - game_shooting_onoff['OnCourt FTA']
game_shooting_onoff['OffCourt OREB'] = game_shooting_onoff['Team OREB'] - game_shooting_onoff['OnCourt OREB']
game_shooting_onoff['OffCourt DREB'] = game_shooting_onoff['Team DREB'] - game_shooting_onoff['OnCourt DREB']
game_shooting_onoff['OffCourt REB'] = game_shooting_onoff['Team REB'] - game_shooting_onoff['OnCourt REB']
game_shooting_onoff['OffCourt AST'] = game_shooting_onoff['Team AST'] - game_shooting_onoff['OnCourt AST']
game_shooting_onoff['OffCourt TOV'] = game_shooting_onoff['Team TOV'] - game_shooting_onoff['OnCourt TOV']
game_shooting_onoff['OffCourt STL'] = game_shooting_onoff['Team STL'] - game_shooting_onoff['OnCourt STL']
game_shooting_onoff['OffCourt BLK'] = game_shooting_onoff['Team BLK'] - game_shooting_onoff['OnCourt BLK']
game_shooting_onoff['OffCourt PF'] = game_shooting_onoff['Team PF'] - game_shooting_onoff['OnCourt PF']
game_shooting_onoff['OffCourt +/-'] = game_shooting_onoff['Team +/-'] - game_shooting_onoff['+/-']
game_shooting_onoff['OffCourt Opp PTS'] = game_shooting_onoff['Opp PTS'] - game_shooting_onoff['OnCourt Opp PTS']
game_shooting_onoff['OffCourt Opp FGM'] = game_shooting_onoff['Opp FGM'] - game_shooting_onoff['OnCourt Opp FGM']
game_shooting_onoff['OffCourt Opp FGA'] = game_shooting_onoff['Opp FGA'] - game_shooting_onoff['OnCourt Opp FGA']
game_shooting_onoff['OffCourt Opp 3PM'] = game_shooting_onoff['Opp 3PM'] - game_shooting_onoff['OnCourt Opp 3PM']
game_shooting_onoff['OffCourt Opp 3PA'] = game_shooting_onoff['Opp 3PA'] - game_shooting_onoff['OnCourt Opp 3PA']
game_shooting_onoff['OffCourt Opp FTM'] = game_shooting_onoff['Opp FTM'] - game_shooting_onoff['OnCourt Opp FTM']
game_shooting_onoff['OffCourt Opp FTA'] = game_shooting_onoff['Opp FTA'] - game_shooting_onoff['OnCourt Opp FTA']
game_shooting_onoff['OffCourt Opp OREB'] = game_shooting_onoff['Opp OREB'] - game_shooting_onoff['OnCourt Opp OREB']
game_shooting_onoff['OffCourt Opp DREB'] = game_shooting_onoff['Opp DREB'] - game_shooting_onoff['OnCourt Opp DREB']
game_shooting_onoff['OffCourt Opp REB'] = game_shooting_onoff['Opp REB'] - game_shooting_onoff['OnCourt Opp REB']
game_shooting_onoff['OffCourt Opp AST'] = game_shooting_onoff['Opp AST'] - game_shooting_onoff['OnCourt Opp AST']
game_shooting_onoff['OffCourt Opp TOV'] = game_shooting_onoff['Opp TOV'] - game_shooting_onoff['OnCourt Opp TOV']
game_shooting_onoff['OffCourt Opp STL'] = game_shooting_onoff['Opp STL'] - game_shooting_onoff['OnCourt Opp STL']
game_shooting_onoff['OffCourt Opp BLK'] = game_shooting_onoff['Opp BLK'] - game_shooting_onoff['OnCourt Opp BLK']

In [515]:
game_shooting_onoff = game_shooting_onoff.rename(columns={'Player_x':'Player'})
game_shooting_onoff = game_shooting_onoff.drop(columns=['Player_y', '4-Man Min', '4-Man PTS',
       '4-Man FGM', '4-Man FGA', '4-Man 3PM', '4-Man 3PA', '4-Man FTM',
       '4-Man FTA', '4-Man OREB', '4-Man DREB', '4-Man REB', '4-Man AST',
       '4-Man TOV', '4-Man STL', '4-Man BLK', '4-Man PF', '4-Man PFD',
       '4-Man +/-', '4-Man Opp FGM', '4-Man Opp FGA', '4-Man Opp 3PM',
       '4-Man Opp 3PA', '4-Man Opp FTM', '4-Man Opp FTA', '4-Man Opp OREB',
       '4-Man Opp DREB', '4-Man Opp REB', '4-Man Opp AST', '4-Man Opp TOV',
       '4-Man Opp STL', '4-Man Opp BLK', '4-Man Opp PTS', 'Team PTS',
       'Team FGM', 'Team FGA', 'Team 3PM', 'Team 3PA', 'Team FTM', 'Team FTA',
       'Team OREB', 'Team DREB', 'Team REB', 'Team AST', 'Team TOV',
       'Team STL', 'Team BLK', 'Team PF', 'Opp PTS', 'Opp FGM', 'Opp FGA',
       'Opp 3PM', 'Opp 3PA', 'Opp FTM', 'Opp FTA', 'Opp OREB', 'Opp DREB',
       'Opp REB', 'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 'Opp PF', 'merge_check'])

In [516]:
player_season_stats = game_shooting_onoff.groupby(['Player', 'Team'])[['GP', 'PTS', 'FGM', 'FGA', '3PM', '3PA', 'FTM', 'FTA',
                'OREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', '+/-', 'Restricted Area FGM', 'Restricted Area FGA', 'In The Paint (Non-RA) FGM',
                'In The Paint (Non-RA) FGA', 'Mid-Range FGM', 'Mid-Range FGA', 'Corner  FGM', 'Corner  FGA', 'Above the Break 3 FGM',
                'Above the Break 3 FGA', 'OnCourt PTS', 'OnCourt FGM', 'OnCourt FGA', 'OnCourt 3PM', 'OnCourt 3PA', 'OnCourt FTM', 'OnCourt FTA', 
                'OnCourt OREB', 'OnCourt DREB', 'OnCourt REB', 'OnCourt AST', 'OnCourt TOV',  'OnCourt STL', 'OnCourt BLK', 'OnCourt PF',
                'OnCourt PFD', 'OnCourt Opp FGM', 'OnCourt Opp FGA', 'OnCourt Opp 3PM', 'OnCourt Opp 3PA', 'OnCourt Opp FTM', 'OnCourt Opp FTA',
                'OnCourt Opp OREB', 'OnCourt Opp DREB', 'OnCourt Opp REB', 'OnCourt Opp AST', 'OnCourt Opp TOV', 'OnCourt Opp STL', 'OnCourt Opp BLK', 
                'OnCourt Opp PTS',  'OffCourt PTS', '4-Man Without Player Min', '4-Man Without Player PTS', '4-Man Without Player FGM', 
                '4-Man Without Player FGA', '4-Man Without Player 3PM', '4-Man Without Player 3PA', '4-Man Without Player FTM', 
                '4-Man Without Player FTA','4-Man Without Player OREB', '4-Man Without Player DREB', '4-Man Without Player REB', 
                '4-Man Without Player AST', '4-Man Without Player TOV', '4-Man Without Player STL', '4-Man Without Player BLK', 
                '4-Man Without Player PF', '4-Man Without Player PFD', '4-Man Without Player +/-', '4-Man Without Player Opp PTS', 
                '4-Man Without Player Opp FGM', '4-Man Without Player Opp FGA', '4-Man Without Player Opp 3PM', '4-Man Without Player Opp 3PA', 
                '4-Man Without Player Opp FTM', '4-Man Without Player Opp FTA', '4-Man Without Player Opp OREB', '4-Man Without Player Opp DREB',
                '4-Man Without Player Opp REB', '4-Man Without Player Opp AST','4-Man Without Player Opp TOV', '4-Man Without Player Opp STL',
                '4-Man Without Player Opp BLK', 'OffCourt FGM', 'OffCourt FGA', 'OffCourt 3PM', 'OffCourt 3PA', 'OffCourt FTM', 'OffCourt FTA', 
                'OffCourt OREB', 'OffCourt DREB', 'OffCourt REB', 'OffCourt AST', 'OffCourt TOV', 'OffCourt STL', 'OffCourt BLK', 'OffCourt +/-',
                'OffCourt PF', 'OffCourt Opp PTS', 'OffCourt Opp FGM', 'OffCourt Opp FGA', 'OffCourt Opp 3PM', 'OffCourt Opp 3PA', 'OffCourt Opp FTM', 
                'OffCourt Opp FTA', 'OffCourt Opp OREB', 'OffCourt Opp DREB', 'OffCourt Opp REB', 'OffCourt Opp AST', 'OffCourt Opp TOV', 
                'OffCourt Opp STL', 'OffCourt Opp BLK']].sum().reset_index()

In [517]:
player_season_averages = game_shooting_onoff.groupby(['Player', 'Team'])[['PTS', 'FGM', 'FGA', '3PM', '3PA', 'FTM', 'FTA',
                'OREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', '+/-', 'Restricted Area FGM', 'Restricted Area FGA', 'In The Paint (Non-RA) FGM',
                'In The Paint (Non-RA) FGA', 'Mid-Range FGM', 'Mid-Range FGA', 'Corner  FGM', 'Corner  FGA', 'Above the Break 3 FGM',
                'Above the Break 3 FGA', '4-Man Without Player +/-', 'OffCourt +/-']].mean().reset_index()

In [518]:
prefixes = ['', 'OnCourt ', 'OnCourt Opp ', '4-Man Without Player ', '4-Man Without Player Opp ', 'OffCourt ', 'OffCourt Opp ']
opp_free_prefixes = ['OnCourt ', '4-Man Without Player ', 'OffCourt ']
for p in prefixes:
    player_season_stats[f'{p}TS%'] = player_season_stats[f'{p}PTS']/(2*player_season_stats[f'{p}FGA']+.88*player_season_stats[f'{p}FTA'])
    player_season_stats[f'{p}TOV%'] = player_season_stats[f'{p}TOV']/(player_season_stats[f'{p}TOV']+player_season_stats[f'{p}FGA']+
                                                                      player_season_stats[f'{p}FTA'])
for p in opp_free_prefixes:
    player_season_stats[f'{p}REB%'] = player_season_stats[f'{p}REB']/(player_season_stats[f'{p}REB']+player_season_stats[f'{p}Opp REB'])

In [519]:
shooting_prefixes = ['', 'Restricted Area ', 'In The Paint (Non-RA) ', 'Mid-Range ']
player_season_averages['3P%'] = player_season_averages['3PM']/player_season_averages['3PA']
for p in shooting_prefixes:
    player_season_averages[f'{p}FG%'] = player_season_averages[f'{p}FGM']/player_season_averages[f'{p}FGA']

In [520]:
player_season_averages.sort_values(inplace=True, ascending=False, by='PTS')

In [521]:
season_league_leaders = player_season_averages.merge(
    player_season_stats[['Player', 'Team', 'GP', 'TS%', 'TOV%', 'OnCourt TS%', 'OnCourt TOV%', 'OnCourt Opp TS%', 'OnCourt Opp TOV%', 
       '4-Man Without Player TS%','4-Man Without Player TOV%', '4-Man Without Player Opp TS%', '4-Man Without Player Opp TOV%', 
        'OffCourt TS%', 'OffCourt TOV%', 'OffCourt Opp TS%', 'OffCourt Opp TOV%', 'OnCourt REB%', 
        '4-Man Without Player REB%', 'OffCourt REB%']],
    on = ['Player', 'Team'],
    indicator = True
)

In [522]:
season_league_leaders = season_league_leaders[season_league_leaders['GP'] >= .5*max(season_league_leaders['GP'])]
season_league_scorers = season_league_leaders[['Player','Team', 'GP', 'PTS', 'OnCourt TS%', '4-Man Without Player TS%', 'OffCourt TS%',
                                               'Restricted Area FG%', 'In The Paint (Non-RA) FG%', 'Mid-Range FG%', '3P%', 'TS%']]
season_league_playmakers=season_league_leaders.sort_values(ascending=False, by='AST')
season_league_playmakers = season_league_playmakers[['Player', 'Team', 'GP', 'AST', 'TOV%', 'OnCourt TOV%', '4-Man Without Player TOV%', 
                                                     'OffCourt TOV%']]
season_league_stealmakers=season_league_leaders.sort_values(ascending=False, by='STL')
season_league_stealmakers = season_league_stealmakers[['Player', 'Team', 'GP', 'STL', 'OnCourt Opp TOV%', '4-Man Without Player Opp TOV%', 
                                                       'OffCourt Opp TOV%']]
season_league_blockers=season_league_leaders.sort_values(ascending=False, by='BLK')
season_league_blockers = season_league_blockers[['Player', 'Team', 'GP', 'BLK', 'OnCourt Opp TS%', '4-Man Without Player Opp TS%', 'OffCourt Opp TS%']]
season_league_rebounders=season_league_leaders.sort_values(ascending=False, by='REB')
season_league_rebounders = season_league_rebounders[['Player', 'Team', 'GP', 'REB', 'OnCourt REB%', '4-Man Without Player REB%', 'OffCourt REB%']]
season_league_plusminus=season_league_leaders.sort_values(ascending=False, by='+/-')
season_league_plusminus['vs. lineup'] = season_league_plusminus['+/-'] - season_league_plusminus['4-Man Without Player +/-']
season_league_plusminus['vs. offcourt'] = season_league_plusminus['+/-'] - season_league_plusminus['OffCourt +/-']
season_league_plusminus=season_league_plusminus.sort_values(ascending=False, by='vs. lineup')
season_league_plusminus= season_league_plusminus[['Player', 'Team', 'GP', '+/-', '4-Man Without Player +/-', 'OffCourt +/-', 'vs. lineup', 
                                                  'vs. offcourt']]

In [523]:
team_opp_stats['Result'] = np.where(team_opp_stats['W'] == 1, 'W', 'L')
team_opp_stats['Game Number'] = team_opp_stats.groupby('Team').cumcount()+1
team_opp_stats['Result'] = np.where(team_opp_stats['W'] == 1, 'W', 'L')
team_opp_stats['Games Ago'] = team_opp_stats.groupby('Team')['Game Number'].transform('max') - team_opp_stats['Game Number'] +1

In [524]:
team_opp_stats = team_opp_stats.sort_values(ascending = True, by = ['Team', 'Date'])
team_opp_stats['Current W'] = team_opp_stats.groupby('Team')['W'].cumsum()
team_opp_stats['Current L'] = team_opp_stats.groupby('Team')['L'].cumsum()
opp_streaks = team_opp_stats[['Date', 'Team', 'Current W', 'Current L']]
opp_streaks = opp_streaks.rename(columns={'Team':'Opponent', 'Current W':'Opp Current W', 'Current L':'Opp Current L'})
team_opp_stats = team_opp_stats.merge(
    opp_streaks,
    on = ['Date', 'Opponent'],
    indicator = False
)
teams = team_opp_stats['Team'].tolist()
results = team_opp_stats['Result'].tolist()
streak = []
for r in range(len(team_opp_stats)):
    if r == 0 or teams[r] != teams[r-1] or results[r] != results[r-1]:
        streak.append(1)
    else:
        streak.append(streak[-1] + 1)
team_opp_stats['Streak number'] = streak
team_opp_stats['Streak'] = team_opp_stats['Result'] + team_opp_stats['Streak number'].astype('str')
current_streak = team_opp_stats.groupby('Team').tail(1)[['Team', 'Streak']]

In [525]:
team_opp_stats = team_opp_stats.merge(
    glossary[['Display', 'NBA']],
    left_on = 'Team',
    right_on = 'Display',
    indicator = False
)
team_opp_stats = team_opp_stats.merge(
    glossary[['Display', 'NBA']],
    left_on = 'Opponent',
    right_on = 'Display',
    indicator = False
)
team_opp_stats.columns

Index(['Date', 'Team', 'Min', 'PTS', 'FGM', 'FGA', '3PM', '3PA', 'FTM', 'FTA',
       'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'PF',
       'Restricted Area FGM', 'Restricted Area FGA',
       'In The Paint (Non-RA) FGM', 'In The Paint (Non-RA) FGA',
       'Mid-Range FGM', 'Mid-Range FGA', 'Corner  FGM', 'Corner  FGA',
       'Above the Break 3 FGM', 'Above the Break 3 FGA', 'OT', 'Team Date',
       'H_A', 'Opponent', 'Opp PTS', 'Opp FGM', 'Opp FGA', 'Opp 3PM',
       'Opp 3PA', 'Opp FTM', 'Opp FTA', 'Opp OREB', 'Opp DREB', 'Opp REB',
       'Opp AST', 'Opp TOV', 'Opp STL', 'Opp BLK', 'Opp PF',
       'Opp Restricted Area FGM', 'Opp Restricted Area FGA',
       'Opp In The Paint (Non-RA) FGM', 'Opp In The Paint (Non-RA) FGA',
       'Opp Mid-Range FGM', 'Opp Mid-Range FGA', 'Opp Corner FGM',
       'Opp Corner FGA', 'Opp Above the Break 3 FGM',
       'Opp Above the Break 3 FGA', '_merge', '+/-', 'W', 'L', 'Result',
       'Game Number', 'Games Ago', 'Current W', 'Current L

In [526]:
team_opp_stats = team_opp_stats.rename(columns={'NBA_x':'Team Initial', 'NBA_y':'Opp Initial'})
team_opp_stats = team_opp_stats.drop(columns=['Display_x', 'Display_y'])

In [527]:
team_opp_stats

,Date,Team,Min,PTS,FGM,FGA,3PM,3PA,FTM,FTA,...,Game Number,Games Ago,Current W,Current L,Opp Current W,Opp Current L,Streak number,Streak,Team Initial,Opp Initial
0,2026-10-21,Atlanta,240.0,132,44,90,14,36,30,36,...,1,14,1,0,0,1,1,W1,ATL,ORL
1,2026-10-23,Atlanta,240.0,128,48,95,18,45,14,22,...,2,13,2,0,1,1,2,W2,ATL,CHA
2,2026-10-24,Atlanta,240.1,114,43,96,13,49,15,21,...,3,12,2,1,3,0,1,L1,ATL,HOU
3,2026-10-26,Atlanta,240.1,110,42,98,11,39,15,21,...,4,11,2,2,4,0,2,L2,ATL,HOU
4,2026-10-28,Atlanta,239.9,117,46,96,16,38,9,14,...,5,10,2,3,1,4,3,L3,ATL,MIA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,2026-11-09,Washington,240.2,111,41,89,9,36,20,28,...,11,5,4,7,7,3,4,L4,WAS,DET
406,2026-11-11,Washington,240.0,101,37,90,11,35,16,22,...,12,4,4,8,4,8,5,L5,WAS,IND
407,2026-11-12,Washington,290.2,140,50,102,14,37,26,29,...,13,3,4,9,8,3,6,L6,WAS,DET
408,2026-11-15,Washington,240.1,119,40,90,15,37,24,34,...,14,2,4,10,11,3,7,L7,WAS,HOU


In [528]:
five_stat_summary = team_season_long[['Team', 'W','L', 'TS%', 'Opp TS%', 'TOV%', 'Opp TOV%', 'REB%']]

In [529]:
five_stat_streak = five_stat_summary.merge(
    current_streak[['Team', 'Streak']],
    on = 'Team',
    indicator=False)

In [530]:
team_season_long['Restricted Area FG%'] = team_season_long['Restricted Area FGM']/team_season_long['Restricted Area FGA']
team_season_long['In The Paint (Non-RA) FG%'] = team_season_long['In The Paint (Non-RA) FGM']/team_season_long['In The Paint (Non-RA) FGA']
team_season_long['Mid-Range FG%'] = team_season_long['Mid-Range FGM']/team_season_long['Mid-Range FGA']
team_season_long['3P%'] = team_season_long['3PM']/team_season_long['3PA']
team_season_long['FT%'] = team_season_long['FTM']/team_season_long['FTA']
shooting_summary = team_season_long[['Team', 'TS%', 'Restricted Area FG%', 'In The Paint (Non-RA) FG%', 'Mid-Range FG%', '3P%', 'FT%']]

In [531]:
shooting_summary = shooting_summary.rename(columns={'Restricted Area FG%':'Restricted FG%', 'In The Paint (Non-RA) FG%':'Non-RA Paint FG%'})

In [532]:
shooting_summary = shooting_summary.sort_values(ascending=False, by='TS%')

In [533]:
team_season_long['Opp Restricted Area FG%'] = team_season_long['Opp Restricted Area FGM']/team_season_long['Opp Restricted Area FGA']
team_season_long['Opp In The Paint (Non-RA) FG%'] = team_season_long['Opp In The Paint (Non-RA) FGM']/team_season_long['Opp In The Paint (Non-RA) FGA']
team_season_long['Opp Mid-Range FG%'] = team_season_long['Opp Mid-Range FGM']/team_season_long['Opp Mid-Range FGA']
team_season_long['Opp 3P%'] = team_season_long['Opp 3PM']/team_season_long['Opp 3PA']
team_season_long['Opp FT%'] = team_season_long['Opp FTM']/team_season_long['Opp FTA']
opp_shooting_summary = team_season_long[['Team', 'Opp TS%', 'Opp Restricted Area FG%', 'Opp In The Paint (Non-RA) FG%', 'Opp Mid-Range FG%', 
                                         'Opp 3P%', 'Opp FT%']]
opp_shooting_summary = opp_shooting_summary.rename(columns={'Opp Restricted Area FG%':'Opp Restricted FG%', 
                                                        'Opp In The Paint (Non-RA) FG%':'Opp Non-RA Paint FG%'})
opp_shooting_summary = opp_shooting_summary.sort_values(ascending=True, by='Opp TS%')

In [534]:
team_season_long['GP'] = team_season_long['W'] + team_season_long['L']
team_season_long['STL'] = team_season_long['STL']/team_season_long['GP']
team_season_long['Opp STL'] = team_season_long['Opp STL']/team_season_long['GP']
tov_summary = team_season_long[['Team', 'TOV%', 'Opp STL']]
tov_summary = tov_summary.sort_values(ascending=True, by='TOV%')
opp_tov_summary = team_season_long[['Team', 'Opp TOV%', 'STL']]
opp_tov_summary = opp_tov_summary.sort_values(ascending=False, by='Opp TOV%')
reb_summary = team_season_long[['Team', 'REB%']]
reb_summary = reb_summary.sort_values(ascending = False, by ='REB%')

In [535]:
# or pick the row with the max date IDXMAX how to - latest = team_opp_stats.loc[team_opp_stats.groupby('Team')['Date'].idxmax()]
game_shooting_onoff['Points Rank'] = game_shooting_onoff.groupby(['Team', 'Date'])['PTS'].rank(method='first', ascending=False)
first_place_scorer = game_shooting_onoff.index[game_shooting_onoff['Points Rank'] == 1]
top_scorers = game_shooting_onoff.loc[first_place_scorer, ['Team', 'Date', 'Player', 'PTS', 'FGM', 'FGA', 'FTM', 'FTA', '3PM', '3PA', 
                                                                 'REB', 'AST', 'TOV', 'STL', 'BLK']]
game_shooting_onoff['Rebounds Rank'] = game_shooting_onoff.groupby(['Team', 'Date'])['REB'].rank(method='first', ascending=False)
first_place_rebounder = game_shooting_onoff.index[game_shooting_onoff['Rebounds Rank'] == 1]
top_rebounders = game_shooting_onoff.loc[first_place_rebounder, ['Team', 'Date', 'Player', 'PTS', 'FGM', 'FGA', 'FTM', 'FTA', '3PM', '3PA', 
                                                                 'REB', 'AST', 'TOV', 'STL', 'BLK']]
game_shooting_onoff['Assists Rank'] = game_shooting_onoff.groupby(['Team', 'Date'])['AST'].rank(method='first', ascending=False)
first_place_assister = game_shooting_onoff.index[game_shooting_onoff['Assists Rank'] == 1]
top_assisters = game_shooting_onoff.loc[first_place_assister, ['Team', 'Date', 'Player', 'PTS', 'FGM', 'FGA', 'FTM', 'FTA', '3PM', '3PA', 
                                                                 'REB', 'AST', 'TOV', 'STL', 'BLK']]
game_shooting_onoff['Steals Rank'] = game_shooting_onoff.groupby(['Team', 'Date'])['STL'].rank(method='first', ascending=False)
first_place_stealer = game_shooting_onoff.index[game_shooting_onoff['Steals Rank'] == 1]
top_stealers = game_shooting_onoff.loc[first_place_stealer, ['Team', 'Date', 'Player', 'PTS', 'FGM', 'FGA', 'FTM', 'FTA', '3PM', '3PA', 
                                                                 'REB', 'AST', 'TOV', 'STL', 'BLK']]

In [536]:
top_scorers['Description'] = (top_scorers['Player']+': '+top_scorers['PTS'].astype('str')+' PTS'+
                              np.where(top_scorers['PTS'] < 18, '', ' ('+top_scorers['FGM'].astype('str')+'-'+top_scorers['FGA'].astype('str')+' FG'+
                              np.where(((top_scorers['FTA']<8) & (top_scorers['3PA']<6)), ')', 
                            np.where(top_scorers['FTA']<8, ', '+top_scorers['3PM'].astype('str')+'-'+top_scorers['3PA'].astype('str')+' 3P)',
                            np.where(top_scorers['3PA']<6, ', '+top_scorers['FTM'].astype('str')+'-'+top_scorers['FTA'].astype('str')+' FT)',
                            ', '+top_scorers['3PM'].astype('str')+'-'+top_scorers['3PA'].astype('str')+' 3P, '+
                            top_scorers['FTM'].astype('str')+'-'+top_scorers['FTA'].astype('str')+' FT)'))))+
                             np.where(top_scorers['REB'] > 5, ', '+top_scorers['REB'].astype('str')+' REB', '')+
                             np.where(top_scorers['AST'] > 4, ', '+top_scorers['AST'].astype('str')+' AST', '')+
                             np.where(top_scorers['TOV'] > 3, ', '+top_scorers['TOV'].astype('str')+' TOV', '')+
                             np.where(top_scorers['STL'] > 1, ', '+top_scorers['STL'].astype('str')+' STL', '')+
                             np.where(top_scorers['BLK'] > 1, ', '+top_scorers['BLK'].astype('str')+' BLK', ''))
top_rebounders['Description'] = (top_rebounders['Player']+': '+top_rebounders['PTS'].astype('str')+' PTS'+
                               np.where(top_rebounders['PTS'] < 18, '',' ('+top_rebounders['FGM'].astype('str')+'-'+top_rebounders['FGA'].astype('str')+' FG'+
                             np.where(((top_rebounders['FTA']<8) & (top_rebounders['3PA']<6)), ')', 
                            np.where(top_rebounders['FTA']<8, ', '+top_rebounders['3PM'].astype('str')+'-'+top_rebounders['3PA'].astype('str')+' 3P)',
                            np.where(top_rebounders['3PA']<6, ', '+top_rebounders['FTM'].astype('str')+'-'+top_rebounders['FTA'].astype('str')+' FT)',
                            ', '+top_rebounders['3PM'].astype('str')+'-'+top_rebounders['3PA'].astype('str')+' 3P, '+
                            top_rebounders['FTM'].astype('str')+'-'+top_rebounders['FTA'].astype('str')+' FT)'))))+
                             np.where(top_rebounders['REB'] > 5, ', '+top_rebounders['REB'].astype('str')+' REB', '')+
                             np.where(top_rebounders['AST'] > 4, ', '+top_rebounders['AST'].astype('str')+' AST', '')+
                             np.where(top_rebounders['TOV'] > 3, ', '+top_rebounders['TOV'].astype('str')+' TOV', '')+
                             np.where(top_rebounders['STL'] > 1, ', '+top_rebounders['STL'].astype('str')+' STL', '')+
                             np.where(top_rebounders['BLK'] > 1, ', '+top_rebounders['BLK'].astype('str')+' BLK', ''))
top_assisters['Description'] = (top_assisters['Player']+': '+top_assisters['PTS'].astype('str')+' PTS'+
                              np.where(top_assisters['PTS'] < 18, '',' ('+top_assisters['FGM'].astype('str')+'-'+top_assisters['FGA'].astype('str')+' FG'+
                              np.where(((top_assisters['FTA']<8) & (top_assisters['3PA']<6)), ')', 
                            np.where(top_assisters['FTA']<8, ', '+top_assisters['3PM'].astype('str')+'-'+top_assisters['3PA'].astype('str')+' 3P)',
                            np.where(top_assisters['3PA']<6, ', '+top_assisters['FTM'].astype('str')+'-'+top_assisters['FTA'].astype('str')+' FT)',
                            ', '+top_assisters['3PM'].astype('str')+'-'+top_assisters['3PA'].astype('str')+' 3P, '+
                            top_assisters['FTM'].astype('str')+'-'+top_assisters['FTA'].astype('str')+' FT)'))))+
                             np.where(top_assisters['REB'] > 5, ', '+top_assisters['REB'].astype('str')+' REB', '')+
                             np.where(top_assisters['AST'] > 4, ', '+top_assisters['AST'].astype('str')+' AST', '')+
                             np.where(top_assisters['TOV'] > 3, ', '+top_assisters['TOV'].astype('str')+' TOV', '')+
                             np.where(top_assisters['STL'] > 1, ', '+top_assisters['STL'].astype('str')+' STL', '')+
                             np.where(top_assisters['BLK'] > 1, ', '+top_assisters['BLK'].astype('str')+' BLK', ''))
top_stealers['Description'] = (top_stealers['Player']+': '+top_stealers['PTS'].astype('str')+' PTS'+
                              np.where(top_stealers['PTS'] < 18, '',' ('+top_stealers['FGM'].astype('str')+'-'+top_stealers['FGA'].astype('str')+' FG'+
                              np.where(((top_stealers['FTA']<8) & (top_stealers['3PA']<6)), ')', 
                            np.where(top_stealers['FTA']<8, ', '+top_stealers['3PM'].astype('str')+'-'+top_stealers['3PA'].astype('str')+' 3P)',
                            np.where(top_stealers['3PA']<6, ', '+top_stealers['FTM'].astype('str')+'-'+top_stealers['FTA'].astype('str')+' FT)',
                            ', '+top_stealers['3PM'].astype('str')+'-'+top_stealers['3PA'].astype('str')+' 3P, '+
                            top_stealers['FTM'].astype('str')+'-'+top_stealers['FTA'].astype('str')+' FT)'))))+
                             np.where(top_stealers['REB'] > 5, ', '+top_stealers['REB'].astype('str')+' REB', '')+
                             np.where(top_stealers['AST'] > 4, ', '+top_stealers['AST'].astype('str')+' AST', '')+
                             np.where(top_stealers['TOV'] > 3, ', '+top_stealers['TOV'].astype('str')+' TOV', '')+
                             np.where(top_stealers['STL'] > 1, ', '+top_stealers['STL'].astype('str')+' STL', '')+
                             np.where(top_stealers['BLK'] > 1, ', '+top_stealers['BLK'].astype('str')+' BLK', ''))

In [537]:
top_scorers = top_scorers.add_prefix('Top Scorer - ')
top_rebounders = top_rebounders.add_prefix('Top Rebounder - ')
top_assisters = top_assisters.add_prefix('Top Assister - ')
top_stealers = top_stealers.add_prefix('Top Stealer - ')

In [538]:
team_opp_stats_performers = team_opp_stats.merge(
    top_scorers[['Top Scorer - Date', 'Top Scorer - Team', 'Top Scorer - Player', 'Top Scorer - Description', 'Top Scorer - PTS']],
    left_on = ['Date', 'Team'],
    right_on = ['Top Scorer - Date', 'Top Scorer - Team'],
    indicator=False)
team_opp_stats_performers = team_opp_stats_performers.merge(
    top_rebounders[['Top Rebounder - Date', 'Top Rebounder - Team', 'Top Rebounder - Player', 'Top Rebounder - Description', 'Top Rebounder - REB']],
    left_on = ['Date', 'Team'],
    right_on = ['Top Rebounder - Date', 'Top Rebounder - Team'],
    indicator=False)
team_opp_stats_performers = team_opp_stats_performers.merge(
    top_assisters[['Top Assister - Date', 'Top Assister - Team', 'Top Assister - Player', 'Top Assister - Description', 'Top Assister - AST']],
    left_on = ['Date', 'Team'],
    right_on = ['Top Assister - Date', 'Top Assister - Team'],
    indicator=False)
team_opp_stats_performers = team_opp_stats_performers.merge(
    top_stealers[['Top Stealer - Date', 'Top Stealer - Team', 'Top Stealer - Player', 'Top Stealer - Description', 'Top Stealer - STL']],
    left_on = ['Date', 'Team'],
    right_on = ['Top Stealer - Date', 'Top Stealer - Team'],
    indicator=False)

In [539]:
team_opp_stats_performers['TS%'] = team_opp_stats_performers['PTS']/(2*team_opp_stats_performers['FGA'] +
                                                                     .88*team_opp_stats_performers['FTA'])
team_opp_stats_performers['Opp TS%'] = team_opp_stats_performers['Opp PTS']/(2*team_opp_stats_performers['Opp FGA'] +
                                                                     .88*team_opp_stats_performers['Opp FTA'])
team_opp_stats_performers['Net TS%'] = team_opp_stats_performers['TS%'] - team_opp_stats_performers['Opp TS%']
team_opp_stats_performers['TOV%'] = team_opp_stats_performers['TOV']/(team_opp_stats_performers['TOV'] + team_opp_stats_performers['FGA'] +
                                                                      .44*team_opp_stats_performers['FTA'])
team_opp_stats_performers['Opp TOV%'] = team_opp_stats_performers['Opp TOV']/(team_opp_stats_performers['Opp TOV'] + 
                                                                              team_opp_stats_performers['Opp FGA'] +
                                                                      .44*team_opp_stats_performers['Opp FTA'])
team_opp_shooting_prefixes = ['3P', 'Opp 3P', 'FG', 'Opp FG', 'FT', 'Opp FT', 'Restricted Area FG', 'Opp Restricted Area FG',
                              'In The Paint (Non-RA) FG', 'Opp In The Paint (Non-RA) FG', 'Mid-Range FG', 'Opp Mid-Range FG']
for p in team_opp_shooting_prefixes:
    team_opp_stats_performers[f'{p}%'] = team_opp_stats_performers[f'{p}M']/team_opp_stats_performers[f'{p}A']
team_opp_stats_performers['Net TOV%'] = team_opp_stats_performers['Opp TOV%'] - team_opp_stats_performers['TOV%']
team_opp_stats_performers['REB%'] = team_opp_stats_performers['REB']/(team_opp_stats_performers['REB'] + team_opp_stats_performers['Opp REB'])
team_opp_stats_performers['Net REB%'] = (team_opp_stats_performers['REB%']-.5)*2
team_opp_stats_performers['TS% Logits'] = team_opp_stats_performers['Net TS%']*108.6121
team_opp_stats_performers['TOV% Logits'] = team_opp_stats_performers['Net TOV%']*114.6224
team_opp_stats_performers['REB% Logits'] = team_opp_stats_performers['Net REB%']*69.65554
team_opp_stats_performers['TS% Logit Split'] = team_opp_stats_performers['TS% Logits'] / (team_opp_stats_performers['TS% Logits'] +
                                                                                          team_opp_stats_performers['TOV% Logits'] +
                                                                                          team_opp_stats_performers['REB% Logits'])
team_opp_stats_performers['TOV% Logit Split'] = team_opp_stats_performers['TOV% Logits'] / (team_opp_stats_performers['TS% Logits'] +
                                                                                          team_opp_stats_performers['TOV% Logits'] +
                                                                                          team_opp_stats_performers['REB% Logits'])
team_opp_stats_performers['REB% Logit Split'] = team_opp_stats_performers['REB% Logits'] / (team_opp_stats_performers['TS% Logits'] +
                                                                                          team_opp_stats_performers['TOV% Logits'] +
                                                                                          team_opp_stats_performers['REB% Logits'])

In [540]:
game_shooting_onoff = game_shooting_onoff.sort_values(ascending=[True, False, False, False], by=['Date', 'PTS', 'TFP', '+/-'])

In [541]:
game_shooting_onoff['Points Rank'] = game_shooting_onoff.groupby(['Team', 'Date'])['PTS'].rank(method='first', ascending=False)
second_place_scorer = game_shooting_onoff.index[game_shooting_onoff['Points Rank'] == 2]
second_scorers = game_shooting_onoff.loc[second_place_scorer, ['Team', 'Date', 'Player', 'PTS', 'FGM', 'FGA', 'FTM', 'FTA', '3PM', '3PA', 
                                                                 'REB', 'AST', 'TOV', 'STL', 'BLK']]
second_scorers['Description'] = (second_scorers['Player']+': '+second_scorers['PTS'].astype('str')+' PTS'+
                               np.where(second_scorers['PTS'] < 18, '', ' ('+second_scorers['FGM'].astype('str')+'-'+second_scorers['FGA'].astype('str')+' FG'+
                             np.where(((second_scorers['FTA']<8) & (second_scorers['3PA']<6)), ')', 
                            np.where(second_scorers['FTA']<8, ', '+second_scorers['3PM'].astype('str')+'-'+second_scorers['3PA'].astype('str')+' 3P)',
                            np.where(second_scorers['3PA']<6, ', '+second_scorers['FTM'].astype('str')+'-'+second_scorers['FTA'].astype('str')+' FT)',
                            ', '+second_scorers['3PM'].astype('str')+'-'+second_scorers['3PA'].astype('str')+' 3P, '+
                            second_scorers['FTM'].astype('str')+'-'+second_scorers['FTA'].astype('str')+' FT)'))))+
                             np.where(second_scorers['REB'] > 5, ', '+second_scorers['REB'].astype('str')+' REB', '')+
                             np.where(second_scorers['AST'] > 4, ', '+second_scorers['AST'].astype('str')+' AST', '')+
                             np.where(second_scorers['TOV'] > 3, ', '+second_scorers['TOV'].astype('str')+' TOV', '')+
                             np.where(second_scorers['STL'] > 1, ', '+second_scorers['STL'].astype('str')+' STL', '')+
                             np.where(second_scorers['BLK'] > 1, ', '+second_scorers['BLK'].astype('str')+' BLK', ''))
second_scorers = second_scorers.add_prefix('Second Scorer - ')
team_opp_stats_performers = team_opp_stats_performers.merge(
    second_scorers[['Second Scorer - Date', 'Second Scorer - Team', 'Second Scorer - Player', 'Second Scorer - Description', 'Second Scorer - PTS']],
    left_on = ['Date', 'Team'],
    right_on = ['Second Scorer - Date', 'Second Scorer - Team'],
    indicator=False)
game_shooting_onoff['Rebound Rank'] = game_shooting_onoff.groupby(['Team', 'Date'])['REB'].rank(method='first', ascending=False)
second_place_rebounder = game_shooting_onoff.index[game_shooting_onoff['Rebound Rank'] == 2]
second_rebounders = game_shooting_onoff.loc[second_place_rebounder, ['Team', 'Date', 'Player', 'PTS', 'FGM', 'FGA', 'FTM', 'FTA', '3PM', '3PA', 
                                                                 'REB', 'AST', 'TOV', 'STL', 'BLK']]
second_rebounders['Description'] = (second_rebounders['Player']+': '+second_rebounders['PTS'].astype('str')+' PTS'+
                              np.where(second_rebounders['PTS'] < 18, '', ' ('+second_rebounders['FGM'].astype('str')+'-'+second_rebounders['FGA'].astype('str')+' FG'+
                              np.where(((second_rebounders['FTA']<8) & (second_rebounders['3PA']<6)), ')', 
                            np.where(second_rebounders['FTA']<8, ', '+second_rebounders['3PM'].astype('str')+'-'+second_rebounders['3PA'].astype('str')+' 3P)',
                            np.where(second_rebounders['3PA']<6, ', '+second_rebounders['FTM'].astype('str')+'-'+second_rebounders['FTA'].astype('str')+' FT)',
                            ', '+second_rebounders['3PM'].astype('str')+'-'+second_rebounders['3PA'].astype('str')+' 3P, '+
                            second_rebounders['FTM'].astype('str')+'-'+second_rebounders['FTA'].astype('str')+' FT)'))))+
                             np.where(second_rebounders['REB'] > 5, ', '+second_rebounders['REB'].astype('str')+' REB', '')+
                             np.where(second_rebounders['AST'] > 4, ', '+second_rebounders['AST'].astype('str')+' AST', '')+
                             np.where(second_rebounders['TOV'] > 3, ', '+second_rebounders['TOV'].astype('str')+' TOV', '')+
                             np.where(second_rebounders['STL'] > 1, ', '+second_rebounders['STL'].astype('str')+' STL', '')+
                             np.where(second_rebounders['BLK'] > 1, ', '+second_rebounders['BLK'].astype('str')+' BLK', ''))
second_rebounders = second_rebounders.add_prefix('Second Rebounder - ')
team_opp_stats_performers = team_opp_stats_performers.merge(
    second_rebounders[['Second Rebounder - Date', 'Second Rebounder - Team', 'Second Rebounder - Player', 'Second Rebounder - Description', 'Second Rebounder - REB']],
    left_on = ['Date', 'Team'],
    right_on = ['Second Rebounder - Date', 'Second Rebounder - Team'],
    indicator=False)
game_shooting_onoff['Assist Rank'] = game_shooting_onoff.groupby(['Team', 'Date'])['AST'].rank(method='first', ascending=False)
second_place_assister = game_shooting_onoff.index[game_shooting_onoff['Assist Rank'] == 2]
second_assisters = game_shooting_onoff.loc[second_place_assister, ['Team', 'Date', 'Player', 'PTS', 'FGM', 'FGA', 'FTM', 'FTA', '3PM', '3PA', 
                                                                 'REB', 'AST', 'TOV', 'STL', 'BLK']]
second_assisters['Description'] = (second_assisters['Player']+': '+second_assisters['PTS'].astype('str')+' PTS'+
                              np.where(second_assisters['PTS'] < 18, '', ' ('+second_assisters['FGM'].astype('str')+'-'+second_assisters['FGA'].astype('str')+' FG'+
                              np.where(((second_assisters['FTA']<8) & (second_assisters['3PA']<6)), ')', 
                            np.where(second_assisters['FTA']<8, ', '+second_assisters['3PM'].astype('str')+'-'+second_assisters['3PA'].astype('str')+' 3P)',
                            np.where(second_assisters['3PA']<6, ', '+second_assisters['FTM'].astype('str')+'-'+second_assisters['FTA'].astype('str')+' FT)',
                            ', '+second_assisters['3PM'].astype('str')+'-'+second_assisters['3PA'].astype('str')+' 3P, '+
                            second_assisters['FTM'].astype('str')+'-'+second_assisters['FTA'].astype('str')+' FT)'))))+
                             np.where(second_assisters['REB'] > 5, ', '+second_assisters['REB'].astype('str')+' REB', '')+
                             np.where(second_assisters['AST'] > 4, ', '+second_assisters['AST'].astype('str')+' AST', '')+
                             np.where(second_assisters['TOV'] > 3, ', '+second_assisters['TOV'].astype('str')+' TOV', '')+
                             np.where(second_assisters['STL'] > 1, ', '+second_assisters['STL'].astype('str')+' STL', '')+
                             np.where(second_assisters['BLK'] > 1, ', '+second_assisters['BLK'].astype('str')+' BLK', ''))
second_assisters = second_assisters.add_prefix('Second Assister - ')
team_opp_stats_performers = team_opp_stats_performers.merge(
    second_assisters[['Second Assister - Date', 'Second Assister - Team', 'Second Assister - Player', 'Second Assister - Description', 'Second Assister - AST']],
    left_on = ['Date', 'Team'],
    right_on = ['Second Assister - Date', 'Second Assister - Team'],
    indicator=False)
game_shooting_onoff['Steal Rank'] = game_shooting_onoff.groupby(['Team', 'Date'])['STL'].rank(method='first', ascending=False)
second_place_stealer = game_shooting_onoff.index[game_shooting_onoff['Steal Rank'] == 2]
second_stealers = game_shooting_onoff.loc[second_place_stealer, ['Team', 'Date', 'Player', 'PTS', 'FGM', 'FGA', 'FTM', 'FTA', '3PM', '3PA', 
                                                                 'REB', 'AST', 'TOV', 'STL', 'BLK']]
second_stealers['Description'] = (second_stealers['Player']+': '+second_stealers['PTS'].astype('str')+' PTS'+
                               np.where(second_stealers['PTS'] < 18, '', ' ('+second_stealers['FGM'].astype('str')+'-'+second_stealers['FGA'].astype('str')+' FG'+
                             np.where(((second_stealers['FTA']<8) & (second_stealers['3PA']<6)), ')', 
                            np.where(second_stealers['FTA']<8, ', '+second_stealers['3PM'].astype('str')+'-'+second_stealers['3PA'].astype('str')+' 3P)',
                            np.where(second_stealers['3PA']<6, ', '+second_stealers['FTM'].astype('str')+'-'+second_stealers['FTA'].astype('str')+' FT)',
                            ', '+second_stealers['3PM'].astype('str')+'-'+second_stealers['3PA'].astype('str')+' 3P, '+
                            second_stealers['FTM'].astype('str')+'-'+second_stealers['FTA'].astype('str')+' FT)'))))+
                             np.where(second_stealers['REB'] > 5, ', '+second_stealers['REB'].astype('str')+' REB', '')+
                             np.where(second_stealers['AST'] > 4, ', '+second_stealers['AST'].astype('str')+' AST', '')+
                             np.where(second_stealers['TOV'] > 3, ', '+second_stealers['TOV'].astype('str')+' TOV', '')+
                             np.where(second_stealers['STL'] > 1, ', '+second_stealers['STL'].astype('str')+' STL', '')+
                             np.where(second_stealers['BLK'] > 1, ', '+second_stealers['BLK'].astype('str')+' BLK', ''))
second_stealers = second_stealers.add_prefix('Second Stealer - ')
team_opp_stats_performers = team_opp_stats_performers.merge(
    second_stealers[['Second Stealer - Date', 'Second Stealer - Team', 'Second Stealer - Player', 'Second Stealer - Description', 'Second Stealer - STL']],
    left_on = ['Date', 'Team'],
    right_on = ['Second Stealer - Date', 'Second Stealer - Team'],
    indicator=False)

In [542]:
#ranked_logit_columns = np.argsort(team_opp_stats_performers[logit_columns].to_numpy(), axis=1, kind='stable')
#team_opp_stats_performers['Second Stat'] = np.array(logit_columns)[ranked_logit_columns[:, -2]]
true_shooting = team_opp_stats_performers['TS% Logits']
tov_rate = team_opp_stats_performers['TOV% Logits']
reb_rate = team_opp_stats_performers['REB% Logits']
opp_tov = team_opp_stats_performers['Opp TOV%']
scorer_1 = team_opp_stats_performers['Top Scorer - Player']
scorer_2 = team_opp_stats_performers['Second Scorer - Player']
scorer_1_pts = team_opp_stats_performers['Top Scorer - PTS']
scorer_2_pts = team_opp_stats_performers['Second Scorer - PTS']
scorer_1_desc = team_opp_stats_performers['Top Scorer - Description']
scorer_2_desc = team_opp_stats_performers['Second Scorer - Description']
rebounder_1 = team_opp_stats_performers['Top Rebounder - Player']
rebounder_2 = team_opp_stats_performers['Second Rebounder - Player']
rebounder_1_reb = team_opp_stats_performers['Top Rebounder - REB']
rebounder_2_reb = team_opp_stats_performers['Second Rebounder - REB']
rebounder_1_desc = team_opp_stats_performers['Top Rebounder - Description']
rebounder_2_desc = team_opp_stats_performers['Second Rebounder - Description']
stealer_1 = team_opp_stats_performers['Top Stealer - Player']
stealer_2 = team_opp_stats_performers['Second Stealer - Player']
stealer_1_stl = team_opp_stats_performers['Top Stealer - STL']
stealer_2_stl = team_opp_stats_performers['Second Stealer - STL']
stealer_1_desc = team_opp_stats_performers['Top Stealer - Description']
stealer_2_desc = team_opp_stats_performers['Second Stealer - Description']
assister_1 = team_opp_stats_performers['Top Assister - Player']
assister_2 = team_opp_stats_performers['Second Assister - Player']
assister_1_ast = team_opp_stats_performers['Top Assister - AST']
assister_2_ast = team_opp_stats_performers['Second Assister - AST']
assister_1_desc = team_opp_stats_performers['Top Assister - Description']
assister_2_desc = team_opp_stats_performers['Second Assister - Description']
plus_minus = team_opp_stats_performers['+/-']

In [543]:
#logit_columns = ['TS% Logits', 'TOV% Logits', 'REB% Logits']
#before = team_opp_stats_performers[logit_columns].copy()
#order = np.argsort(team_opp_stats_performers[logit_columns].to_numpy(), axis=1, kind='stable')
#np.maximum takes only two arguments
team_opp_stats_performers['Top Stat Line'] = np.where(team_opp_stats_performers['Result'] == 'L', '', 
                                            np.where(true_shooting > np.maximum(tov_rate, reb_rate), scorer_1_desc,
                                                     np.where(tov_rate > np.maximum(true_shooting, reb_rate),
                                                              np.where((opp_tov > .16) & (stealer_1_stl > 2), stealer_1_desc,
                                                                       np.where(assister_1_ast > 6, assister_1_desc, scorer_1_desc)),
                                                              np.where(rebounder_1_reb > 9 , rebounder_1_desc, scorer_1_desc))))
team_opp_stats_performers['Top Performer 1'] = team_opp_stats_performers['Top Stat Line'].str.split(':').str[0]
top_performer = team_opp_stats_performers['Top Performer 1']
fallback = np.where(top_performer != scorer_1, scorer_1_desc, scorer_2_desc)
ts_tov_branch = np.select([(top_performer != scorer_1) & (scorer_1_pts > 23),
                            (top_performer != scorer_2) & (scorer_2_pts > 23),
                            (assister_1 != top_performer) & (assister_1_ast > 6) & (opp_tov < .16),
                            (assister_2 != top_performer) & (assister_2_ast > 6) & (opp_tov < .16),
                            (stealer_1 != top_performer) & (stealer_1_stl > 2),
                            (stealer_2 != top_performer) & (stealer_2_stl > 2)],
                           [scorer_1_desc, scorer_2_desc, assister_1_desc, assister_2_desc, stealer_1_desc, stealer_2_desc],
                            default = fallback)
reb_branch = np.where((top_performer != rebounder_1) & (rebounder_1_reb > 9), rebounder_1_desc,
                      np.where((top_performer != rebounder_2) & (rebounder_2_reb > 12), rebounder_2_desc,
                               np.where(top_performer != scorer_1, scorer_1_desc, scorer_2_desc)))
team_opp_stats_performers['Top Stat Line 2'] = np.where(team_opp_stats_performers['Result'] == 'L', '',
                                                        np.where(reb_rate <= np.maximum(tov_rate, true_shooting),
                                                                 ts_tov_branch, reb_branch))
team_opp_stats_performers['Top Performer 2'] = team_opp_stats_performers['Top Stat Line 2'].str.split(':').str[0]
team_opp_stats_performers['Losing Team Top Stat Line'] = np.where(team_opp_stats_performers['Result'] == 'W', '',
                                                                  np.where((true_shooting > np.maximum(tov_rate, reb_rate)) | (plus_minus < -15), 
                                                                           scorer_1_desc,
                                                                           np.where(tov_rate > np.maximum(true_shooting, reb_rate), 
                                                                                    np.where((opp_tov > .16) & (stealer_1_stl > 2), stealer_1_desc,
                                                                                             np.where(((assister_1_ast > 6) & (scorer_1_pts < 25)), 
                                                                                                      assister_1_desc, scorer_1_desc)),
                                                                                    np.where(rebounder_1_reb > 9 , rebounder_1_desc, scorer_1_desc))))
team_opp_stats_performers['Losing Top Performer 1'] = team_opp_stats_performers['Losing Team Top Stat Line'].str.split(':').str[0]

In [544]:
opponent_metrics = team_opp_stats_performers[['Date', 'Team', 'Losing Team Top Stat Line', 'Losing Top Performer 1', 'Streak']]
team_opp_stats_performers.drop(columns=['_merge', 'Top Scorer - Team', 'Top Scorer - Date', 'Top Rebounder - Date', 'Top Rebounder - Team',
                                        'Top Assister - Date', 'Top Assister - Team', 'Top Stealer - Date', 'Top Stealer - Team',
                                        'Second Scorer - Date', 'Second Scorer - Team', 'Second Rebounder - Date', 'Second Rebounder - Team',
                                        'Second Assister - Date', 'Second Assister - Team', 'Second Stealer - Date', 'Second Stealer - Team', 
                                        'Losing Team Top Stat Line', 'Losing Top Performer 1'], inplace=True)

In [545]:
team_opp_stats_performers = team_opp_stats_performers.merge(
    opponent_metrics,
    left_on = ['Date', 'Opponent'],
    right_on = ['Date', 'Team'],
    indicator = False)

In [546]:
team_opp_stats_performers.drop(columns=['Team_y'], inplace=True)

In [547]:
team_opp_stats_performers.rename(columns={'Streak_y':'Opp Streak', 'Streak_x':'Streak', 'Team_x':'Team'}, inplace=True)

In [548]:
team_opp_stats_performers['Game Notes - 3P'] = np.where(team_opp_stats_performers['Result']=='L', '', 
                                                        np.where((team_opp_stats_performers['3PA'] > 50) | (team_opp_stats_performers['3PA'] < 20) |
                                                                (team_opp_stats_performers['3P%'] >= .5) | (team_opp_stats_performers['3P%'] <= .2),
                                                                team_opp_stats_performers['3PM'].astype('str')
                                                                 +'-'+team_opp_stats_performers['3PA'].astype('str')+' 3P', ''))
team_opp_stats_performers['Game Notes - Opp 3P'] = np.where(team_opp_stats_performers['Result']=='L', '',
                                                            np.where((team_opp_stats_performers['Opp 3PA'] > 50) | 
                                                                     (team_opp_stats_performers['Opp 3PA'] < 20) |
                                                                     (team_opp_stats_performers['Opp 3P%'] >= .5) | 
                                                                     (team_opp_stats_performers['Opp 3P%'] <= .2),
                                                                     team_opp_stats_performers['Opp 3PM'].astype('str')
                                                                     +'-'+ team_opp_stats_performers['Opp 3PA'].astype('str')+' 3P', ''))
team_opp_stats_performers['Game Notes - STL'] = np.where(team_opp_stats_performers['Result']=='L', '',
                                                         np.where((team_opp_stats_performers['STL'] > 12) | (team_opp_stats_performers['STL'] < 4),
                                                                  team_opp_stats_performers['STL'].astype('str')+' STL', ''))
team_opp_stats_performers['Game Notes - Opp STL'] = np.where(team_opp_stats_performers['Result']=='L', '',
                                                             np.where((team_opp_stats_performers['Opp STL'] > 12) | 
                                                                      (team_opp_stats_performers['Opp STL'] < 4),
                                                                      team_opp_stats_performers['Opp STL'].astype('str')+' STL', ''))
team_opp_stats_performers['Game Notes - FTA'] = np.where(team_opp_stats_performers['Result']=='L', '',
                                                         np.where((team_opp_stats_performers['FTA'] > 40) | (team_opp_stats_performers['FTA'] < 11) |
                                                                  (team_opp_stats_performers['FT%'] > .875) | (team_opp_stats_performers['FT%'] < .6),
                                                                  team_opp_stats_performers['FTM'].astype('str')+'-'+
                                                                  team_opp_stats_performers['FTA'].astype('str')+' FT', ''))
team_opp_stats_performers['Game Notes - Opp FTA'] = np.where(team_opp_stats_performers['Result']=='L', '',
                                                             np.where((team_opp_stats_performers['Opp FTA'] > 40) | 
                                                                      (team_opp_stats_performers['Opp FTA'] < 11) |
                                                                      (team_opp_stats_performers['Opp FT%'] > .875) | 
                                                                      (team_opp_stats_performers['Opp FT%'] < .6),
                                                                      team_opp_stats_performers['Opp FTM'].astype('str')+'-'+
                                                                      team_opp_stats_performers['Opp FTA'].astype('str')+' FT', ''))
team_opp_stats_performers['Game Notes - Restricted'] = np.where(team_opp_stats_performers['Result']=='L', '',
                                                                np.where((team_opp_stats_performers['Restricted Area FGA'] > 39) |
                                                                         (team_opp_stats_performers['Restricted Area FGA'] < 11) |
                                                                         (team_opp_stats_performers['Restricted Area FG%'] > .875) |
                                                                         (team_opp_stats_performers['Restricted Area FG%'] < .45),
                                                                         team_opp_stats_performers['Restricted Area FGM'].astype('str')+'-'+
                                                                         team_opp_stats_performers['Restricted Area FGA'].astype('str')+
                                                                         ' Restricted FG', ''))
team_opp_stats_performers['Game Notes - Opp Restricted'] = np.where(team_opp_stats_performers['Result']=='L', '',
                                                                    np.where((team_opp_stats_performers['Opp Restricted Area FGA'] > 39) |
                                                                             (team_opp_stats_performers['Opp Restricted Area FGA'] < 11) |
                                                                             (team_opp_stats_performers['Opp Restricted Area FG%'] > .875) |
                                                                             (team_opp_stats_performers['Opp Restricted Area FG%'] < .45),
                                                                             team_opp_stats_performers['Opp Restricted Area FGM'].astype('str')+'-'+
                                                                             team_opp_stats_performers['Opp Restricted Area FGA'].astype('str')+
                                                                             ' Restricted FG', ''))
team_opp_stats_performers['Team Game Notes'] = (np.where((team_opp_stats_performers['Game Notes - 3P'] == '') &
                                                         (team_opp_stats_performers['Game Notes - STL'] == '') &
                                                         (team_opp_stats_performers['Game Notes - FTA'] == '') &
                                                         (team_opp_stats_performers['Game Notes - Restricted'] == ''), '',
                                                         team_opp_stats_performers['Team Initial']+': '+
                                                         team_opp_stats_performers['Game Notes - 3P']+
                                                         np.where(((team_opp_stats_performers['Game Notes - 3P'] != '')&
                                                                   (team_opp_stats_performers['Game Notes - STL'] != '')), ', ', '')+ 
                                                         team_opp_stats_performers['Game Notes - STL'] +
                                                         np.where((((team_opp_stats_performers['Game Notes - 3P'] != '')|
                                                                  (team_opp_stats_performers['Game Notes - STL'] != ''))&
                                                                  (team_opp_stats_performers['Game Notes - FTA'] != '')), ', ', '')+
                                                         team_opp_stats_performers['Game Notes - FTA'] +
                                                         np.where((((team_opp_stats_performers['Game Notes - 3P'] != '')|
                                                                  (team_opp_stats_performers['Game Notes - STL'] != '')|
                                                                  (team_opp_stats_performers['Game Notes - FTA'] != ''))&
                                                                   (team_opp_stats_performers['Game Notes - Restricted'] !='')), ', ', '')+
                                                         team_opp_stats_performers['Game Notes - Restricted']))
team_opp_stats_performers['Opp Game Notes'] = (np.where((team_opp_stats_performers['Game Notes - Opp 3P'] == '') &
                                                         (team_opp_stats_performers['Game Notes - Opp STL'] == '') &
                                                         (team_opp_stats_performers['Game Notes - Opp FTA'] == '') &
                                                         (team_opp_stats_performers['Game Notes - Opp Restricted'] == ''), '',
                                                         team_opp_stats_performers['Opp Initial']+': '+
                                                         team_opp_stats_performers['Game Notes - Opp 3P']+
                                                         np.where(((team_opp_stats_performers['Game Notes - Opp 3P'] != '')&
                                                                   (team_opp_stats_performers['Game Notes - Opp STL'] != '')), ', ', '')+ 
                                                         team_opp_stats_performers['Game Notes - Opp STL'] +
                                                         np.where((((team_opp_stats_performers['Game Notes - Opp 3P'] != '')|
                                                                  (team_opp_stats_performers['Game Notes - Opp STL'] != ''))&
                                                                  (team_opp_stats_performers['Game Notes - Opp FTA'] != '')), ', ', '')+
                                                         team_opp_stats_performers['Game Notes - Opp FTA'] +
                                                         np.where((((team_opp_stats_performers['Game Notes - Opp 3P'] != '')|
                                                                  (team_opp_stats_performers['Game Notes - Opp STL'] != '')|
                                                                  (team_opp_stats_performers['Game Notes - Opp FTA'] != ''))&
                                                                   (team_opp_stats_performers['Game Notes - Opp Restricted'] !='')), ', ', '')+
                                                         team_opp_stats_performers['Game Notes - Opp Restricted']))
team_opp_stats_performers['Game Notes'] = team_opp_stats_performers['Team Game Notes'] + np.where((team_opp_stats_performers['Team Game Notes'] !='') &
                                                                                                  (team_opp_stats_performers['Opp Game Notes'] != ''),
                                                                                                  ', ', '')+team_opp_stats_performers['Opp Game Notes']

In [549]:
yesterday = date.today() - timedelta(days = 1)
yesterday = pd.Timestamp(yesterday)
yesterday_games = team_opp_stats_performers[team_opp_stats_performers['Date'] == yesterday]
yesterday_games = yesterday_games[yesterday_games['Result'] == 'W']

In [558]:
tonight = date.today()
tonight = pd.Timestamp(tonight)
tonight_games = schedule[schedule['new_date'] == tonight]
tonight_games = tonight_games[['Start (ET)', 'Visitor/Neutral', 'Home/Neutral']]

In [550]:
yesterday_games['Team Scores'] = yesterday_games['Team']+np.where(yesterday_games['H_A']==1, ' (H)', '')+'\n' + yesterday_games['Opponent']+np.where(yesterday_games['H_A']==0, ' (H)', '')
yesterday_games['Points/Opp'] = yesterday_games['PTS'].astype('str') +'\n'+yesterday_games['Opp PTS'].astype('str')
yesterday_games['Records'] = yesterday_games['Current W'].astype('str')+'-'+yesterday_games['Current L'].astype('str')+'\n'+yesterday_games['Opp Current W'].astype('str')+'-'+yesterday_games['Opp Current L'].astype('str')
yesterday_games['Streaks'] = yesterday_games['Streak'] + '\n' + yesterday_games['Opp Streak']
yesterday_games['TS%/Opp'] = yesterday_games['TS%'].map('{:.3f}'.format) + '\n' + yesterday_games['Opp TS%'].map('{:.3f}'.format)
yesterday_games['TOV%/Opp'] = yesterday_games['TOV%'].map('{:.3f}'.format) + '\n' + yesterday_games['Opp TOV%'].map('{:.3f}'.format)
yesterday_games['REB/Opp'] = yesterday_games['REB'].astype('str') + '\n' + yesterday_games['Opp REB'].astype('str')
yesterday_games['Top Stat Line/Opp'] = yesterday_games['Top Stat Line']+'; '+yesterday_games['Top Stat Line 2']+ '\n' + yesterday_games['Losing Team Top Stat Line']+ np.where(yesterday_games['Game Notes'] != '', '; '+yesterday_games['Game Notes'], '')
yesterday_games['Total W'] = (yesterday_games['Current W']+yesterday_games['Opp Current W'])
yesterday_games = yesterday_games.sort_values(ascending=False, by='Total W')
full_summary = yesterday_games[['Team Scores', 'Points/Opp', 'OT', 'Records', 'Streaks', 'TS%/Opp', 'TOV%/Opp', 'REB/Opp', 'Top Stat Line/Opp']]
full_summary = full_summary.rename(columns={'Team Scores':'', 'Points/Opp':'PTS', 'Records':'  ', 'Streaks':' ', 'TS%/Opp':'TS%','TOV%/Opp':'TOV',
                                            'REB/Opp':'REB', 'Top Stat Line/Opp':'Top Performer'})
(full_summary.reset_index(drop=True).style
    .set_properties(**{'white-space': 'pre-wrap',
                       'text-align': 'left',
                       'vertical-align': 'middle'}))

,,PTS,OT,,,TS%,TOV,REB,Top Performer
0,Minnesota (H) Portland,106 98,,10-4 9-4,W1 L1,0.509 0.482,0.111 0.113,53 49,"Terrence Shannon Jr: 34 PTS (8-11 FG, 15-16 FT), 6 AST, 4 TOV; Rudy Gobert: 18 PTS (5-10 FG, 8-14 FT), 15 REB, 2 BLK Deni Avdija: 27 PTS (11-19 FG), 8 REB, 5 TOV"
1,Miami (H) Houston,128 110,,8-6 11-4,W5 L1,0.621 0.523,0.080 0.117,40 52,"Giannis Antetokounmpo: 20 PTS (8-11 FG), 13 REB, 6 AST, 3 BLK; Davion Mitchell: 16 PTS, 10 AST, 4 STL Alperen Sengun: 23 PTS (8-11 FG), 7 REB"
2,Orlando Boston (H),115 105,,6-7 11-3,W4 L1,0.618 0.539,0.123 0.101,46 40,"Desmond Bane: 24 PTS (8-13 FG, 3-8 3P), 8 REB, 5 AST; Franz Wagner: 21 PTS (8-21 FG), 8 REB, 9 AST, 4 TOV Jayson Tatum: 24 PTS (9-23 FG, 3-10 3P), 12 REB, 5 TOV, 2 STL; BOS: 20-56 3P, 9-17 FT"
3,Philadelphia Brooklyn (H),135 123,,12-3 3-10,W3 L2,0.645 0.597,0.095 0.097,47 38,"Jaylen Brown: 31 PTS (9-19 FG, 3-8 3P, 10-13 FT), 13 REB; Joel Embiid: 37 PTS (13-22 FG, 9-10 FT), 2 BLK Julius Randle: 33 PTS (13-20 FG), 6 REB, 2 STL; BKN: 28-41 Restricted FG"
4,Memphis Washington (H),136 120,,10-4 4-11,W3 L8,0.664 0.595,0.149 0.106,48 31,"Zach Edey: 23 PTS (10-15 FG), 10 REB; Cameron Boozer: 29 PTS (10-21 FG, 7-13 3P), 7 REB, 7 AST, 5 TOV Anthony Davis: 23 PTS (8-15 FG, 7-8 FT), 2 BLK"
5,New Orleans (H) Golden State,115 94,,5-10 8-6,W1 L3,0.535 0.484,0.085 0.141,57 50,"Jeremiah Fears: 19 PTS (6-13 FG), 10 REB, 10 AST; Herbert Jones: 8 PTS, 3 STL Yaxel Lendeborg: 28 PTS (9-23 FG, 2-9 3P, 8-8 FT), 8 REB, 4 TOV, 2 BLK; GSW: 27-30 FT"
6,Phoenix Denver (H),117 105,,6-7 7-5,W1 L1,0.591 0.575,0.124 0.165,51 35,"Oso Ighodaro: 8 PTS, 11 REB; Devin Booker: 20 PTS (5-14 FG, 9-10 FT), 8 AST Jamal Murray: 28 PTS (9-19 FG, 1-6 3P, 9-9 FT), 2 STL"
7,Cleveland (H) Chicago,126 115,,8-7 3-10,W2 L1,0.593 0.562,0.158 0.142,51 37,"Jarrett Allen: 13 PTS, 14 REB, 4 TOV; Donovan Mitchell: 39 PTS (16-25 FG), 3 STL Norman Powell: 27 PTS (10-18 FG, 0-6 3P, 7-11 FT), 6 AST, 4 TOV; CLE: 14 STL, CHI: 14 STL"


In [559]:
tonight_games

,Start (ET),Visitor/Neutral,Home/Neutral
205,7:30p,New York Knicks,Toronto Raptors
206,8:00p,Portland Trail Blazers,Minnesota Timberwolves
207,8:00p,Detroit Pistons,San Antonio Spurs
208,10:00p,Charlotte Hornets,Los Angeles Lakers
209,11:00p,Utah Jazz,Los Angeles Clippers
